In [0]:
SET spark.databricks.execution.timeout = 600;

In [0]:
SELECT
  event_id,
  event_timestamp,
  _metadata.file_modification_time AS landed_at
FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
LIMIT 5;

In [0]:
WITH first_arrival AS (
  SELECT
    event_id,
    CAST(event_timestamp AS TIMESTAMP) AS event_ts,
    _metadata.file_modification_time AS landed_at
  FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
  QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY _metadata.file_modification_time) = 1
),
latency AS (
  SELECT (unix_millis(landed_at) - unix_millis(event_ts)) / 1000.0 AS latency_sec
  FROM first_arrival
)
SELECT
  COUNT(*) AS events,
  ROUND(AVG(latency_sec), 2) AS avg_sec,
  ROUND(PERCENTILE(latency_sec, 0.5), 2) AS p50_sec,
  ROUND(PERCENTILE(latency_sec, 0.95), 2) AS p95_sec,
  ROUND(MAX(latency_sec), 2) AS max_sec
FROM latency;

In [0]:
SELECT
  COUNT(*) AS events,
  ROUND(AVG((unix_millis(_eventhub_enqueued_timestamp)
           - unix_millis(CAST(event_timestamp AS TIMESTAMP))) / 1000.0), 2) AS avg_to_broker_sec,
  ROUND(AVG((unix_millis(_ingestion_timestamp)
           - unix_millis(_eventhub_enqueued_timestamp)) / 1000.0), 2)       AS avg_broker_to_table_sec,
  ROUND(AVG((unix_millis(_ingestion_timestamp)
           - unix_millis(CAST(event_timestamp AS TIMESTAMP))) / 1000.0), 2) AS avg_end_to_end_sec
FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.eventhub_youtube_bronze;